In [1]:
# ================================================================
# 🚀 FINAL DEPLOYMENT SCRIPT: TUNE, COMPARE & SAVE
# ================================================================

import pandas as pd
import numpy as np
import joblib
import shap
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

# -----------------------------------------------------------
# 1️⃣ LOAD DATA & SPLIT
# -----------------------------------------------------------
print("⏳ Loading Data...")
X = pd.read_csv('df_ready_for_clustering.csv')
if 'Unnamed: 0' in X.columns:
    X = X.drop(columns=['Unnamed: 0'])

y_data = pd.read_csv('ICU_LOS_Validation.csv')
y_los = y_data['los']

# Target: >= 3 days = Long Stay (1)
y_binary = (y_los >= 3).astype(int)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

# SMOTE Balancing
print("⚖️ Balancing training data...")
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# -----------------------------------------------------------
# 2️⃣ HYPERPARAMETER TUNING (On All Features First)
# -----------------------------------------------------------
print("🔍 Tuning Full Model... (Please wait)")

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 50, 70],
    'max_depth': [-1, 10, 15, 20],
    'min_child_samples': [10, 20, 30],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

lgbm = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

random_search = RandomizedSearchCV(
    lgbm, param_dist, n_iter=20, scoring='f1', cv=3, verbose=1, random_state=42, n_jobs=-1
)
random_search.fit(X_train_sm, y_train_sm)
best_full_model = random_search.best_estimator_
tuned_params = random_search.best_params_

print(f"✅ Best Parameters Found: {tuned_params}")

# -----------------------------------------------------------
# 3️⃣ EXTRACT TOP 10 FEATURES (Based on Tuned Model)
# -----------------------------------------------------------
print("📊 Extracting Top 10 Features...")
explainer = shap.TreeExplainer(best_full_model)
shap_values = explainer.shap_values(X_train_sm)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

mean_abs_shap =  np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': X_train_sm.columns,
    'Mean_SHAP': mean_abs_shap
}).sort_values(by='Mean_SHAP', ascending=False)

# Force Paper Features if you want, OR use these dynamic Top 10
# (Uncomment next line to strictly use Paper features instead)
top_10_features = ['op_duration_min', 'intraop_rocu', 'optype_Colorectal', 'optype_Vascular', 'aline1', 'preop_alb', 'intraop_crystalloid', 'intraop_uo', 'cormack_Easy', 'optype_Stomach']
# top_10_features = shap_importance.head(10)['Feature'].tolist()

print(f"✅ Final Top 10 Features: {top_10_features}")

# -----------------------------------------------------------
# 4️⃣ TRAIN COMPACT MODEL (With Tuned Params)
# -----------------------------------------------------------
print("🔄 Training Final Compact Model...")

X_train_top10 = X_train_sm[top_10_features]
X_test_top10  = X_test[top_10_features]

final_compact_model = LGBMClassifier(
    **tuned_params,  # Use same best parameters
    random_state=42, n_jobs=-1, verbose=-1
)
final_compact_model.fit(X_train_top10, y_train_sm)

# -----------------------------------------------------------
# 5️⃣ FINAL COMPARISON & METRICS
# -----------------------------------------------------------
def get_metrics(model, X, y_true, name):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba)
    }

metrics_full = get_metrics(best_full_model, X_test, y_test, "Full Tuned Model")
metrics_compact = get_metrics(final_compact_model, X_test_top10, y_test, "Compact Tuned Model")

print("\n📊 Final Results Comparison:")
display(pd.DataFrame([metrics_full, metrics_compact]).round(4))

# -----------------------------------------------------------
# 6️⃣ SAVE FOR BACKEND
# -----------------------------------------------------------
print("\n💾 Saving 'los_model.pkl' for Deployment...")
model_data = {
    "model": final_compact_model,
    "features": top_10_features,
    "threshold": 3
}
joblib.dump(model_data, 'los_model.pkl')
print("🎉 Deployment Ready! Download 'los_model.pkl' now.")

⏳ Loading Data...
⚖️ Balancing training data...
🔍 Tuning Full Model... (Please wait)
Fitting 3 folds for each of 20 candidates, totalling 60 fits
✅ Best Parameters Found: {'subsample': 1.0, 'num_leaves': 50, 'n_estimators': 200, 'min_child_samples': 20, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
📊 Extracting Top 10 Features...


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


✅ Final Top 10 Features: ['op_duration_min', 'intraop_rocu', 'optype_Colorectal', 'optype_Vascular', 'aline1', 'preop_alb', 'intraop_crystalloid', 'intraop_uo', 'cormack_Easy', 'optype_Stomach']
🔄 Training Final Compact Model...

📊 Final Results Comparison:


,Model,Accuracy,F1 Score,Recall,ROC-AUC
0,Full Tuned Model,0.9390,0.9657,0.9632,0.9557
1,Compact Tuned Model,0.9124,0.9507,0.9482,0.9306



💾 Saving 'los_model.pkl' for Deployment...
🎉 Deployment Ready! Download 'los_model.pkl' now.


In [3]:
import pandas as pd
#ye raw data hai
Y = pd.read_csv('Cleaned_Clinical_data_with_noscaling.csv')

print(Y.columns)

Index(['bmi', 'asa', 'emop', 'preop_htn', 'preop_dm', 'preop_hb', 'preop_plt',
       'preop_pt', 'preop_aptt', 'preop_na', 'preop_k', 'preop_gluc',
       'preop_alb', 'preop_ast', 'preop_alt', 'preop_bun', 'preop_cr',
       'tubesize', 'iv2', 'aline1', 'cline1', 'intraop_ebl', 'intraop_uo',
       'intraop_rbc', 'intraop_ffp', 'intraop_crystalloid', 'intraop_colloid',
       'intraop_ppf', 'intraop_mdz', 'intraop_ftn', 'intraop_rocu',
       'intraop_vecu', 'intraop_eph', 'intraop_phe', 'intraop_epi',
       'intraop_ca', 'op_duration_min', 'ane_duration_min',
       'age_Children__0_14_years_', 'age_Youth__15_24_years_',
       'age_Young_Adults__25_44_years_', 'age_Middle_Age__45_59_years_',
       'age_Elderly__60_74_years_', 'age_Senior__75__years_',
       'preop_ecg_abnormal', 'sex_F', 'sex_M', 'optype_Breast',
       'optype_Colorectal', 'optype_Hepatic', 'optype_Major resection',
       'optype_Minor resection', 'optype_Others', 'optype_Stomach',
       'optype_Thyroid', 'op

In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. SETUP: LOAD DATA & RE-CREATE SCALER
# ==========================================
print("⚙️ Loading resources to build Scaler...")

# Humen Raw Data chahiye taaki Scaler ko pata ho Mean/Std kya hai
# (Make sure filenames match exactly what you have)
df_raw = Y

# Model load karein taaki humein pata chale kaunse features chahiye
model_data = joblib.load('los_model.pkl')
model = model_data['model']
top_features = model_data['features'] # Yeh wohi Top 10 list hai

print(f"🎯 Model expects these features: {top_features}")

# --- Data Prep for Scaler (Important!) ---
# Humein Raw data ko waisa hi banana hai jaise training ke waqt tha
# Agar aapne training se pehle dummies banaye the, toh yahan bhi banane honge
df_raw_processed = pd.get_dummies(df_raw)

# Check karein ki saare Top 10 columns maujood hain ya nahi
missing_cols = [col for col in top_features if col not in df_raw_processed.columns]
if missing_cols:
    print(f"⚠️ Warning: These columns are missing in raw data: {missing_cols}")
    # Fix: Missing columns ko 0 fill kar dete hain (safe bet for binaries)
    for col in missing_cols:
        df_raw_processed[col] = 0

# Sirf Top 10 columns ko select karke Scaler fit karein
scaler = StandardScaler()
scaler.fit(df_raw_processed[top_features])

print("✅ Scaler is ready! Now taking input...")

# ==========================================
# 2. DEFINE YOUR TEST CASE (RAW NUMBERS)
# ==========================================
# Yahan aap naye patient ka RAW data daliye
new_patient = {
    'op_duration_min': 0,      # 6 hours!
        'intraop_rocu': 20,         # High drug dose
        'optype_Colorectal': 1,      # High Risk Surgery
        'optype_Vascular': 0,
        'aline1': 0,
        'preop_alb': 2.2,            # Dangerously low
        'intraop_crystalloid': 100, # Huge fluid loss
        'intraop_uo': 10,            # Kidney struggling
        'cormack_Easy': 1,           # Difficult intubation
        'optype_Stomach': 0
}

# Convert dict to DataFrame
input_df = pd.DataFrame([new_patient])

# Ensure columns are in correct order (Same as training)
input_df = input_df[top_features]

# ==========================================
# 3. SCALE & PREDICT
# ==========================================

# 1. Scale the raw input
input_scaled = scaler.transform(input_df)

# 2. Predict
probability = model.predict_proba(input_scaled)[0][1]
prediction = model.predict(input_scaled)[0]

print("\n" + "="*40)
print(f"🩺 PATIENT PREDICTION REPORT")
print("="*40)
print(f"Risk Probability: {probability:.2%}")

if prediction == 1:
    print("🔴 RESULT: HIGH RISK (Likely LOS >= 3 Days)")
    print("   Action: Plan for ICU bed or extended monitoring.")
else:
    print("🟢 RESULT: LOW RISK (Likely LOS < 3 Days)")
    print("   Action: Standard ward recovery path.")
print("="*40)

⚙️ Loading resources to build Scaler...
🎯 Model expects these features: ['op_duration_min', 'intraop_rocu', 'optype_Colorectal', 'optype_Vascular', 'aline1', 'preop_alb', 'intraop_crystalloid', 'intraop_uo', 'cormack_Easy', 'optype_Stomach']
✅ Scaler is ready! Now taking input...

🩺 PATIENT PREDICTION REPORT
Risk Probability: 98.39%
🔴 RESULT: HIGH RISK (Likely LOS >= 3 Days)
   Action: Plan for ICU bed or extended monitoring.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. SETUP: SCALER SIRF CONTINUOUS KE LIYE
# ==========================================
print("⚙️ Preparing Pipeline...")

# Model Load
model_data = joblib.load('los_model.pkl')
model = model_data['model']
# Yeh wo list hai jo Model ko chahiye (Total 10)
all_features = model_data['features']

# --- Features ko 2 hisson mein baato ---
continuous_vars = [
    'op_duration_min', 'intraop_rocu', 'preop_alb',
    'intraop_crystalloid', 'intraop_uo'
]

binary_vars = [
    'optype_Colorectal', 'optype_Vascular', 'aline1',
    'cormack_Easy', 'optype_Stomach'
]

# Load Raw Data just to fit the Scaler on Continuous vars
# (Make sure 'Y' is your Raw Data DataFrame loaded previously)
df_raw = Y

# Sirf Continuous columns par Scaler fit karein
scaler = StandardScaler()
scaler.fit(df_raw[continuous_vars])

print("✅ Scaler ready for Continuous variables only.")

# ==========================================
# 2. DEFINE 5 TEST CASES
# ==========================================
test_cases = [
    # CASE 1: The "Safe" Patient 🟢
    # Short surgery, Healthy Albumin (4.5), Good Urine Output
    {
        'id': 'Case 1 (Safe)',
        'desc': 'Healthy patient, short surgery (1 hr)',
        'op_duration_min': 60,
        'intraop_rocu': 10,
        'preop_alb': 4.5,
        'intraop_crystalloid': 500,
        'intraop_uo': 400,
        'optype_Colorectal': 0, 'optype_Vascular': 0, 'aline1': 0, 'cormack_Easy': 1, 'optype_Stomach': 0
    },

    # CASE 2: The "Borderline" Patient 🟡
    # Medium surgery (3 hrs), Slightly Low Albumin, Stomach Surgery
    {
        'id': 'Case 2 (Borderline)',
        'desc': 'Medium duration, Stomach Op, mild risk',
        'op_duration_min': 120,      # 2 Hours
        'intraop_rocu': 40,
        'preop_alb': 4.8,            # Excellent Health (High Albumin protects)
        'intraop_crystalloid': 1000,
        'intraop_uo': 250,
        'optype_Colorectal': 0, 'optype_Vascular': 0, 'aline1': 0, 'cormack_Easy': 1, 'optype_Stomach': 0
    },

    # CASE 3: The "Critical" Patient 🔴
    # (Your original case) Long duration, Low Albumin, Colorectal
    {
        'id': 'Case 3 (Critical)',
        'desc': 'Long surgery (6 hrs), Colorectal, Low Albumin',
        'op_duration_min': 360,
        'intraop_rocu': 100,
        'preop_alb': 2.2,
        'intraop_crystalloid': 3500,
        'intraop_uo': 40,
        'optype_Colorectal': 1, 'optype_Vascular': 0, 'aline1': 1, 'cormack_Easy': 0, 'optype_Stomach': 0
    },

    # CASE 4: The "Vascular" Patient 🔴
    # Vascular surgery is inherently high risk, even if duration is medium
    {
        'id': 'Case 4 (Vascular)',
        'desc': 'Vascular Surgery, High Blood Loss Risk',
        'op_duration_min': 240,
        'intraop_rocu': 60,
        'preop_alb': 3.2,
        'intraop_crystalloid': 2000,
        'intraop_uo': 80,
        'optype_Colorectal': 0, 'optype_Vascular': 1, 'aline1': 1, 'cormack_Easy': 0, 'optype_Stomach': 0
    },

    # CASE 5: The "Old/Weak but Short Op" 🟡
    # Very Low Albumin (Risk) but very short surgery (Safety factor) - Confusing case
    {
        'id': 'Case 5 (Mixed)',
        'desc': 'Very weak patient but short surgery',
        'op_duration_min': 45,
        'intraop_rocu': 20,
        'preop_alb': 2.0,  # Very Low
        'intraop_crystalloid': 300,
        'intraop_uo': 100,
        'optype_Colorectal': 0, 'optype_Vascular': 0, 'aline1': 0, 'cormack_Easy': 1, 'optype_Stomach': 0
    }
]

# ==========================================
# 3. LOOP THROUGH CASES & PREDICT
# ==========================================
print("\n" + "="*60)
print(f"{'CASE ID':<20} | {'PROBABILITY':<10} | {'PREDICTION'}")
print("="*60)

for patient in test_cases:
    # 1. Input ko DataFrame banao
    # (Hum sirf relevant columns uthayenge taaki extra keys 'id/desc' ignore ho jayein)
    input_df = pd.DataFrame([patient])

    # 2. Continuous ko Scale karo
    # Note: Hum wahi 'continuous_vars' use kar rahe hain jo upar define kiye
    input_df[continuous_vars] = scaler.transform(input_df[continuous_vars])

    # 3. Binary Vars ko touch mat karo (Already 0/1 hain)

    # 4. Columns ko wahi order do jo Model chahta hai
    final_input = input_df[all_features]

    # 5. Predict
    probability = model.predict_proba(final_input)[0][1]
    prediction = model.predict(final_input)[0]

    # Print Result
    risk_label = "🔴 HIGH RISK" if prediction == 1 else "🟢 LOW RISK"
    print(f"{patient['id']:<20} | {probability:.2%}     | {risk_label}")
    # print(f"   ({patient['desc']})") # Optional description

print("="*60)

⚙️ Preparing Pipeline...
✅ Scaler ready for Continuous variables only.

CASE ID              | PROBABILITY | PREDICTION
Case 1 (Safe)        | 31.60%     | 🟢 LOW RISK
Case 2 (Borderline)  | 99.74%     | 🔴 HIGH RISK
Case 3 (Critical)    | 99.99%     | 🔴 HIGH RISK
Case 4 (Vascular)    | 99.98%     | 🔴 HIGH RISK
Case 5 (Mixed)       | 99.00%     | 🔴 HIGH RISK
